In [0]:
# =============================================================================
# E-Commerce Analytics Pipeline — Databricks Free Edition
# =============================================================================
# Description : End-to-end Spark data engineering pipeline using two source
#               files (orders.json + customers.csv) with full transformations:
#               Joins, GroupBy, Case/When, Window Functions, Pivot, Delta write
#
# Architecture: Bronze → Silver → Gold (Medallion)
# Author      : Data Engineering Template
# Compatibility: Databricks Community Edition | Apache Spark 3.x | Python 3.8+
# =============================================================================

import json
import random
from datetime import datetime, timedelta

# from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType, DateType,BooleanType
)
from pyspark.sql.window import Window


# =============================================================================
# INITIALISE SPARK SESSION
# =============================================================================


# =============================================================================
# STEP 1 — CREATE SAMPLE SOURCE FILES
# =============================================================================
# Generates:  /dbfs/FileStore/ecommerce/orders.json    (200 orders, nested items)
#             /dbfs/FileStore/ecommerce/customers.csv   (50 customers)

def generate_source_files():
    """Write synthetic orders.json and customers.csv to DBFS."""

    import os
    os.makedirs("/Volumes/bigdata2/my_voloum/use-case2/ecommerce", exist_ok=True)

    random.seed(42)  # reproducible data

    # ── orders.json (newline-delimited JSON) ─────────────────────────────────
    statuses        = ["delivered", "shipped", "cancelled", "returned", "pending"]
    categories      = ["Electronics", "Clothing", "Books", "Home", "Sports"]
    payment_methods = ["credit_card", "debit_card", "UPI", "netbanking", "wallet"]
    cities          = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad"]

    orders = []
    for i in range(1, 201):
        num_items  = random.randint(1, 4)
        items      = []
        for j in range(num_items):
            qty   = random.randint(1, 5)
            price = round(random.uniform(50, 5000), 2)
            items.append({
                "item_id":      f"ITEM{random.randint(100, 999)}",
                "product_name": f"Product_{random.choice(categories)}_{j + 1}",
                "category":     random.choice(categories),
                "quantity":     qty,
                "unit_price":   price,
                "line_total":   round(qty * price, 2),
            })

        order_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 364))
        orders.append({
            "order_id":       f"ORD{i:05d}",
            "customer_id":    f"CUST{random.randint(1, 50):04d}",
            "order_date":     order_date.strftime("%Y-%m-%d"),
            "order_status":   random.choice(statuses),
            "payment_method": random.choice(payment_methods),
            "shipping_city":  random.choice(cities),
            "discount_pct":   random.choice([0, 5, 10, 15, 20]),
            "items":          items,
        })

    with open("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/orders.json", "w") as f:
        for order in orders:
            f.write(json.dumps(order) + "\n")

    # ── customers.csv ─────────────────────────────────────────────────────────
    tiers  = ["Bronze", "Silver", "Gold", "Platinum"]
    states = ["Maharashtra", "Delhi", "Karnataka", "Tamil Nadu", "Telangana"]

    csv_lines = ["customer_id,name,email,age,gender,city,state,tier,signup_date,is_active"]
    for i in range(1, 51):
        signup = datetime(2021, 1, 1) + timedelta(days=random.randint(0, 800))
        csv_lines.append(
            f"CUST{i:04d},"
            f"Customer_{i},"
            f"cust{i}@email.com,"
            f"{random.randint(22, 60)},"
            f"{random.choice(['M', 'F', 'Other'])},"
            f"{random.choice(cities)},"
            f"{random.choice(states)},"
            f"{random.choice(tiers)},"
            f"{signup.strftime('%Y-%m-%d')},"
            f"{random.choice(['true', 'true', 'true', 'false'])}"
        )

    with open("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/customers.csv", "w") as f:
        f.write("\n".join(csv_lines))

    print(f"orders.json written   → {len(orders)} records")
    print(f" customers.csv written → {len(csv_lines) - 1} records")


generate_source_files()

In [0]:
# ----------------------------------------------------------------------------------------
#  Step 2 -  Bronze Layer insert raw layer 
# ----------------------------------------------------------------------------------------

#  --- 2A. read order.json (nested/newline- delimited)-----------------------------------
from pyspark.sql import functions as F

#  --- Read JSON ---
df_orders_raw = (
    spark.read
        .option("inferSchema", "true")
        .json("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/orders.json")   # ⚠ fixed path
)

print("\n---------------- orders.json schema ----------------")
df_orders_raw.printSchema()


# --- Explode items array ---
df_orders_exploded = (
    df_orders_raw
        .withColumn("item", F.explode("items"))   # explode array
        .select(
            "order_id",
            "customer_id",
            "order_date",
            "order_status",
            "payment_method",
            "shipping_city",
            "discount_pct",
            F.col("item.item_id").alias("item_id"),
            F.col("item.product_name").alias("product_name"),
            F.col("item.category").alias("category"),
            F.col("item.quantity").alias("quantity"),
            F.col("item.unit_price").alias("unit_price"),
            F.col("item.line_total").alias("line_total")
        )
)

display(df_orders_exploded)

In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType() ,True),
    StructField("name", StringType(),    True),
    StructField("email", StringType(),    True),
    StructField("age", IntegerType(),    True),
    StructField("gender", StringType(),  True),
    StructField("city", StringType(),    True),
    StructField("state", StringType(),   True),
    StructField("tier", StringType(),    True),
    StructField("signup_date", DateType(),True),
    StructField("is_active", BooleanType(),True)
])

df_customers_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .schema(customer_schema)
    .csv("/Volumes/bigdata2/my_voloum/use-case2/ecommerce/customers.csv"))

print(f"\n Customers loaded:{df_customers_raw.count} rows")
display(df_customers_raw)

In [0]:
# ---- 3A. /clean orders--------------------------------
df_orders_clean = (
    df_orders_exploded
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("order_quarter", F.quarter("order_date"))

    # net line total discount
    .withColumn(
        "net_line_total",
        F.round(F.col("line_total") * (1 - F.col("discount_pct") / 100), 2)
    )

    # Standard free-text fields
    .withColumn("order_status", F.lower(F.trim(F.col("order_status"))))
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))

    # drop rows missing critical keys
    .dropna(subset=["order_id", "customer_id", "order_date"])  # if we want to remove duplicate rows in specife task then we use subset 

    # remove exact duplicate line items
    .dropDuplicates(["order_id", "item_id"])
)

display(df_orders_clean)

In [0]:
df_customer_clean=(
    df_customers_raw
    .withColumn(
        "age_group",
        F.when(F.col("age") < 30, "18-29")
        .when(F.col("age") < 40, "30-39")
        .when(F.col("age") < 50, "40-49")
        .otherwise("50+"),
    )
    # customer tenure in days since signup
    .withColumn(
        "tenure_days",
        F.datediff(F.current_date(), F.col("signup_date")),
    )
    .dropna(subset=["customer_id"])
    
)

In [0]:
df_silver = (
    df_orders_clean
    .join(
        F.broadcast(df_customer_clean),
        on="customer_id",
        how="inner"
    )

    # Case 1: revenue segment based on line total
    .withColumn(
        "revenue_segment",
        F.when(F.col("net_line_total") >= 10000, "high value")
         .when(F.col("net_line_total") >= 3000, "mid value")
         .when(F.col("net_line_total") >= 500, "low value")
         .otherwise("micro")
    )

    # Case 2: binary flag (is this a revenue order?)
    .withColumn(
        "is_revenue_order",
        F.when(
            F.col("order_status").isin("delivered", "shipped"), 1
        ).otherwise(0)
    )

    # Case 3: effective revenue (cancelled/returned = 0)
    .withColumn(
        "effective_revenue",
        F.when(
            F.col("order_status").isin("cancelled", "returned"), 0
        ).otherwise(F.col("net_line_total"))
    )
)

display(df_silver)

In [0]:
# from pyspark.sql.window import Window

# city_window = Window.partitionBy("shipping_city")

# df_silver = (
#     df_silver
#     .withColumn(
#         "city_total_revenue",
#         F.sum("effective_revenue").over(city_window)
#     )
# )

# display(df_silver)

In [0]:

# category_window = (
#     Window
#     .partitionBy("category")
#     .orderBy("order_date")
#     .rowsBetween(Window.unboundedPreceding, Window.currentRow)
# )

# df_running_sum = (
#     df_silver
#     .withColumn(
#         "running_category_revenue",
#         F.sum("effective_revenue").over(category_window)
#     )
# )

# display(df_running_sum)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# w1: per-customer chronological order
w_cust_date = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
)

# w2: running total per customer
w_cust_running = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# w3: rank within each city + category
w_city_cat = (
    Window
    .partitionBy("shipping_city", "category")
    .orderBy(F.desc("net_line_total"))
)

# w4: rolling 3 rows per customer
w_rolling3 = (
    Window
    .partitionBy("customer_id")
    .orderBy("order_date")
    .rowsBetween(-2, Window.currentRow)
)

df_windowed = (
    df_silver

    # order sequence number
    .withColumn(
        "order_seq_num",
        F.row_number().over(w_cust_date)
    )

    # cumulative spend
    .withColumn(
        "cumulative_spend",
        F.round(F.sum("effective_revenue").over(w_cust_running), 2)
    )

    # rank within city + category
    .withColumn(
        "rank_in_city_cat",
        F.dense_rank().over(w_city_cat)
    )

    # previous order revenue
    .withColumn(
        "prev_order_revenue",
        F.lag("effective_revenue", 1).over(w_cust_date)
    )

    # next order revenue
    .withColumn(
        "next_order_revenue",
        F.lead("effective_revenue", 1).over(w_cust_date)
    )

    # rolling average of last 3 orders
    .withColumn(
        "rolling_avg_3",
        F.round(F.avg("effective_revenue").over(w_rolling3), 2)
    )

    # revenue difference from previous order
    .withColumn(
        "revenue_delta",
        F.round(F.col("effective_revenue") - F.col("prev_order_revenue"), 2)
    )

    # percent rank within customer
    .withColumn(
        "pct_rank_in_customer",
        F.round(F.percent_rank().over(w_cust_date), 3)
    )
)

print("\n-- window function sample (per customer time series) --")

df_windowed.select(
    "customer_id",
    "order_id",
    "order_date",
    "effective_revenue",
    "order_seq_num",
    "cumulative_spend",
    "prev_order_revenue",
    "revenue_delta",
    "rolling_avg_3"
).show(10, truncate=False)